In [3]:
pip install -r requirements.txt

Note: you may need to restart the kernel to use updated packages.


In [1]:
# python notebook to investigate the question classification for nl questions
import json
from langchain_ollama import OllamaLLM
from langchain_core.prompts import ChatPromptTemplate
import os
import pandas as pd


MODEL = "qwen3:8b"  # Specify the model you want to use
llm = OllamaLLM(
    model=MODEL,
    base_url="http://localhost:11434",  # Specify the URL of your Ollama instance
    temperature=0.7,
    num_predict=-1,
    repeat_penalty=1.3,
    repeat_last_n=256,
    num_ctx=8192,
    format="json",  # Ensure the response is in JSON format
)

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a helpful assistant that generates questions based on SPARQL queries."
            "Only return the question, do not include any additional text."
            "when encountering a '~*' in the query with * being a number, assume it is a wildcard"
            "and replace it with a space in the question."
            "for instance ('ROSKOGO~5 VB~5') are 2 separate terms with a wildcard and should be ROSKOGO or VB in the question."
            "Do not use any markdown in the question, just plain text."
            "Generate questions with different levels of specificity based on the query."
            "Also generate questions asking for different aspects of the query, for instance,"
            "if sample, observatory, and event are in the query, you can ask about the sample,"
            "the observatory, or the event, or a combination of them."
            "Make sure to always include all the main entities and all relationships in the question."
            "These will be in the lines where there is onto:fts , FILTER regex and FILTER"
            "focus on including all the values that the sparql query filters on,"
            "give the reponse in format of a dictionary with keys sparql_query and question, like this: "
            "'sparql_query': your_sparql_query, 'question': 'your_question'"
            "do not include any other text in the response.",
        ),
        ("user", "Generate a question for the following SPARQL query: {sparql_query}"),
    ]
)

# load in all_generated_questions.json
def load_and_filter_data(file_path):
    df = pd.read_json(file_path, lines=True)

    # Filter rows where 'question' starts with 'SELECT'
    filtered_question = df[df["question"].str.startswith("{")]

    # Filter rows where 'sparql_select' ends with '...'
    filtered_sparql = df[df["sparql_query"].str.endswith("...")]
    
    # Exclude the filtered rows
    df = df[~df["question"].str.startswith("SELECT")]
    df = df[~df["sparql_query"].str.endswith("...")]

    return df

data_file = "./all_generated_questions.json"
dataset = load_and_filter_data(data_file)
print(f"Loaded {len(dataset)} rows from {data_file}")

ModuleNotFoundError: No module named 'langchain_ollama'